In [1]:
import sqlite3
import pandas as pd
from scipy.spatial import cKDTree
import math

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) 

In [2]:
fn = 'points.sqlite'
points = pd.read_sql('select * from points where not banned', sqlite3.connect(fn))

# activity

In [15]:
import requests
import pandas as pd

overpass_url = "https://overpass-api.de/api/interpreter"
overpass_query = """
[out:json][timeout:25];
nwr["highway"="hitchhiking"];
out meta;
"""

response = requests.post(overpass_url, data={'data': overpass_query})
data = response.json()

# Extract nodes
elements = data['elements']
nodes = [el for el in elements if el['type'] == 'node']

# Convert to DataFrame for easier handling
nodes_df = pd.DataFrame([{
    'id': node['id'],
    'lat': node['lat'],
    'lon': node['lon'],
    'tags': node.get('tags', {}),
    'timestamp': node.get('timestamp'),
    'user': node.get('user'),
    'uid': node.get('uid')
} for node in nodes])

len(nodes_df)

672

In [21]:
nodes_df.tail(200)

,id,lat,lon,tags,timestamp,user,uid
472,12739058848,52.078591,5.141546,"{'bench': 'no', 'highway': 'hitchhiking', 'note': 'Staat met een bord aangegeven', 'shelter': 'no'}",2025-04-08T14:01:19Z,DwarfNebula_,3735696
473,12748988716,51.657097,9.904826,"{'bench': 'yes', 'covered': 'no', 'destination': 'Hardegsen;Nörten-Hardenberg', 'highway': 'hitchhiking', 'min_age': '18', 'shelter': 'no'}",2025-04-28T08:02:37Z,jengelh,1468043
474,12748988718,51.657021,9.905081,"{'bench': 'yes', 'covered': 'no', 'destination': 'Moringen;Northeim', 'highway': 'hitchhiking', 'min_age': '18', 'shelter': 'no'}",2025-04-28T08:02:37Z,jengelh,1468043
475,12763147347,50.088797,11.569999,"{'bench': 'yes', 'covered': 'no', 'highway': 'hitchhiking'}",2025-04-17T21:42:07Z,Rainero,137242
476,12785258828,48.013738,9.501211,"{'amenity': 'bench', 'backrest': 'yes', 'bench': 'yes', 'highway': 'hitchhiking', 'material': 'wood', 'name': 'Mitfahrer Bänkle'}",2025-05-13T13:03:06Z,NVBW_Edit_7,22845345
477,12799976030,45.308313,5.957128,"{'highway': 'hitchhiking', 'lit': 'yes'}",2025-04-30T10:06:28Z,icluf,12970344
478,12818184014,52.137691,13.474836,"{'amenity': 'bench', 'armrest': 'no', 'backrest': 'yes', 'bench': 'yes', 'bin': 'no', 'colour': 'brown', 'covered': 'no', 'direction': '325', 'highway': 'hitchhiking', 'lit': 'no', 'mapillary': '786848489530193; 808001224121509', 'material': 'wood', 'name': 'Mitfahrbank', 'pole': 'yes', 'seats': '2', 'shelter': 'no', 'survey:date': '2025-05-23'}",2025-05-24T16:36:03Z,mueschel,616774
479,12832500529,44.870212,5.596818,"{'highway': 'hitchhiking', 'operator': 'Communauté de communes du Trièves'}",2025-05-12T22:20:23Z,r2d,2210139
480,12836806861,52.276786,9.659332,"{'bench': 'yes', 'destination': 'Weetzen Bahnhof', 'highway': 'hitchhiking', 'name': 'Mitfahrerbank', 'shelter': 'no'}",2025-05-14T12:18:37Z,werbuli,14853477
481,12836916961,52.293752,9.637699,"{'bench': 'yes', 'destination': 'Vörie;Linderte', 'highway': 'hitchhiking', 'name': 'Mitfahrerbank', 'shelter': 'no'}",2025-05-14T13:04:13Z,werbuli,14853477


In [ ]:
tree = cKDTree(points[['lat', 'lon']].values)

distances, indices = tree.query(nodes_df[['lat', 'lon']].values)

# Add nearest node info to points DataFrame
nodes_df['nearest_node_id'] = points.iloc[indices]['id'].values
nodes_df['nearest_node_lat'] = points.iloc[indices]['lat'].values
nodes_df['nearest_node_lon'] = points.iloc[indices]['lon'].values
nodes_df['distance'] = distances

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

In [ ]:
nodes_df['haversine_distance'] = nodes_df.apply(lambda row: haversine(row['lat'], row['lon'], row['nearest_node_lat'], row['nearest_node_lon']) * 1000, axis=1)

In [ ]:
nodes_df = nodes_df.sort_values(by='haversine_distance')

In [ ]:
nodes_df.head(100)